# Define scenarios

Set up which stroke units are available, the proportions of stroke type in the population, and the proportion of people who receieve a transfer for drip-and-ship.

Fixed across scenarios:
+ travel bias for thrombectomy units - if a thrombectomy unit is no more than X minutes farther away than a non-thrombectomy unit, then choose the thrombectomy unit.
+ proportion of LVO

For each scenario, need:
+ scenario name
+ a list of new MT units (affects geoprocessing)
+ proportion of LVO who receive a transfer for thrombectomy (affects combination of results from outcome model)

__Result__: file `scenario_parameters.csv`

## Code setup

In [1]:
import pandas as pd

## Load data

Units and available services:

In [2]:
df_hospitals = pd.read_csv('data/stroke_hospitals_2025.csv')

In [3]:
display(df_hospitals.head(3))

,SSNAP name,Postcode,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
0,John Radcliffe Hospital,OX39DU,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
1,Royal Berkshire Hospital,RG15AN,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
2,Wycombe General Hospital,HP112TT,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


# Fixed parameters

In [4]:
# --- Travel ---
# Allow for a travel bias time (minutes) to MT centres
mt_travel_bias = 0.1

# --- Population ---
# How many patients have LVO (rest have nLVO):
prop_lvo = 0.3

# Scenario parameters

In [5]:
# --- Travel ---
# Allow some drip-ship patients to receive a transfer for thrombectomy.
# Others do not receive thrombectomy.
dict_prop_lvo_dripship_treated_with_mt = {
    '': 1.0,
    'lossydripshipmt': 0.6
}

# --- New MT units ---
# Base case: no new units.
new_mt_units_base = []  # empty on purpose
# Main case: Taunton and Worcester are both new.
new_mt_units_main = ['TA15DA', 'WR51DD']
# Individual case: have only one of these units at once.
# Get postcode for all hospitals with Use_IVT=1 and Use_MT=0
m = ((df_hospitals['Use_IVT'] == 1) &
     (df_hospitals['Use_MT'] == 0) &
     (df_hospitals['Country'] == 'England'))
units_to_test = df_hospitals.loc[m, 'Postcode'].tolist()

Check units to test:

In [6]:
print(len(units_to_test))

print(units_to_test[:5])

73
['RG15AN', 'HP112TT', 'CH21UL', 'L355DR', 'CH495PE']


Gather sets of new units:

In [7]:
options_new_units = [new_mt_units_base, new_mt_units_main] + (
    [[u] for u in units_to_test])
# Track whether these are the individual case:
options_individual = (
    [False] * (len(options_new_units) - len(units_to_test)) +
    [True] * len(units_to_test)
)
# Scenario names:
scenario_names = (['base', 'taunworc'] +
                  [f'only_{u}' for u in units_to_test])

In [8]:
print(options_new_units[:5])
print(options_individual[:5])
print(scenario_names[:5])

[[], ['TA15DA', 'WR51DD'], ['RG15AN'], ['HP112TT'], ['CH21UL']]
[False, False, True, True, True]
['base', 'taunworc', 'only_RG15AN', 'only_HP112TT', 'only_CH21UL']


## Build scenarios

For each combination of a set of new units and a proportion transferred in drip-and-ship, create scenario parameters.

In [9]:
df = pd.DataFrame()

# Variable parameters:
# Start with all unit combinations.
df['scenario_name'] = pd.Series(scenario_names, dtype=str)
df['scenario_geoprocess_name'] = pd.Series(scenario_names, dtype=str)
df['new_mt_units'] = pd.Series(options_new_units, dtype=object)
df['single_unit_changed'] = pd.Series(options_individual, dtype=int)
# Add in proportion 
list_of_dfs = []
for label, p in dict_prop_lvo_dripship_treated_with_mt.items():
    df_here = df.copy()
    df_here['prop_lvo_dripship_treated_with_mt'] = p
    if len(label) > 0:
        df_here['scenario_name'] = df_here['scenario_name'] + '_' + label
    list_of_dfs.append(df_here)
df = pd.concat(list_of_dfs, axis='rows').sort_index(
    kind='stable', ignore_index=True)

# Fixed parameters:
df['mt_travel_bias_minutes'] = mt_travel_bias
df['prop_lvo'] = prop_lvo

In [10]:
display(df.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
0,base,base,[],0,1.0,0.1,0.3
1,base_lossydripshipmt,base,[],0,0.6,0.1,0.3
2,taunworc,taunworc,"[TA15DA, WR51DD]",0,1.0,0.1,0.3
3,taunworc_lossydripshipmt,taunworc,"[TA15DA, WR51DD]",0,0.6,0.1,0.3
4,only_RG15AN,only_RG15AN,[RG15AN],1,1.0,0.1,0.3
5,only_RG15AN_lossydripshipmt,only_RG15AN,[RG15AN],1,0.6,0.1,0.3


## Save scenario parameters

In [11]:
path_to_output = './output/scenario_parameters.csv'
df.to_csv(path_to_output, index=False)

## Load scenarios back in

In [12]:
# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
converters = {
    'new_mt_units': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
}
df_reload = pd.read_csv(path_to_output, converters=converters)

Check that reloaded data is the same as the dataframe we just created:

In [13]:
display(df_reload.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,prop_lvo_dripship_treated_with_mt,mt_travel_bias_minutes,prop_lvo
0,base,base,[],0,1.0,0.1,0.3
1,base_lossydripshipmt,base,[],0,0.6,0.1,0.3
2,taunworc,taunworc,"[TA15DA, WR51DD]",0,1.0,0.1,0.3
3,taunworc_lossydripshipmt,taunworc,"[TA15DA, WR51DD]",0,0.6,0.1,0.3
4,only_RG15AN,only_RG15AN,[RG15AN],1,1.0,0.1,0.3
5,only_RG15AN_lossydripshipmt,only_RG15AN,[RG15AN],1,0.6,0.1,0.3


In [14]:
(df_reload == df).all()

scenario_name                        True
scenario_geoprocess_name             True
new_mt_units                         True
single_unit_changed                  True
prop_lvo_dripship_treated_with_mt    True
mt_travel_bias_minutes               True
prop_lvo                             True
dtype: bool

In [15]:
df_reload['new_mt_units']

0                    []
1                    []
2      [TA15DA, WR51DD]
3      [TA15DA, WR51DD]
4              [RG15AN]
             ...       
145            [BD96RJ]
146            [HX30PW]
147            [HX30PW]
148            [WF14DG]
149            [WF14DG]
Name: new_mt_units, Length: 150, dtype: object

## Conclusion

We have created a file with the parameters that change between each scenario.